In [1]:
import json
from openai import OpenAI
import os
from tqdm.std import tqdm
import re
import json
import ast
import numpy as np
from scipy.stats import ttest_rel
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
Code_set = ["PartnershipProvider", "InfoGive", "InfoSeek", "PartnershipPatient", "InfoGiveSDOH", "SharedDecisionPatient", "Socioemotional/Empathy", "SharedDecisionProvider", "InfoSeekSDOH"]

Sub_Code_set = ["salutation", "SchedulingAppt", "requestsForOpinion", "connection", "statePreferences", "Appreciation/Gratitude", "HealthCareAccessAndQuality", "signoff", "Instruction", "Generalinformation", "ApprovalofDecision", "NeighborhoodAndBuiltEnvironment", "Diagnostics", "carecoordination", "Drugs", "Symptoms", "maintainCommunication", "AcknowledgeError", "activeParticipation/involvement", "inviteCollaboration", "expressOpinions", "ExpressConcern/unease", "EncourageQuestions", "SummarizeConfirmUnderstanding", "Reassurance", "Approval", "Alignment", "SeekingApproval", "checkingUnderstanding", "EconomicStability", "ExploreOptions", "ShareOptions", "Sadness/fear", "Prognosis", "build trust", "checkingUnderstanding/clarification", "acknowledgePatientExpertiseKnowledge", "ApprovalofDecision/Reinforcement", "Approval/Reinforcement", "MakeDecision", "PositiveRemarks"]

In [3]:
def relaxed_match_evaluation_with_semantic(true_entities_list, pred_entities_list):
    """
    Evaluate partial matches of named entities using Relaxed Match, combining full containment logic and Jaccard coefficient
    :param true_entities_list: list of true entities, one per sentence [[str, ...], ...]
    :param pred_entities_list: list of predicted entities, one per sentence [[str, ...], ...]
    :return: Precision, Recall, F1 Score
    """
    true_positive = 0
    false_positive = 0
    false_negative = 0

    # Iterate over the true and predicted entities in each sentence
    for true_entities, pred_entities in zip(true_entities_list, pred_entities_list):
        matched_true = [False] * len(true_entities)
        matched_pred = [False] * len(pred_entities)

        # For each predicted entity, check whether it partially matches a real entity
        for i, pred_entity in tqdm(enumerate(pred_entities)):
            for j, true_entity in enumerate(true_entities):
                user_prompt = get_user_prompt(pred_entity, true_entity)
                response = get_response(user_prompt)
                response = json.loads(response)
                if 'yes' in response.get("similar").lower():
                        true_positive += 1
                        matched_true[j] = True
                        matched_pred[i] = True
        # Counting False Positives and False Negatives
        false_positive += matched_pred.count(False)  # No matching predicted entities found
        false_negative += matched_true.count(False)  # No real entity matched

    # Calculate Precision, Recall, F1 Score
    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
    recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1

In [4]:
def calculate_code(true_codes, pred_codes):
    code_mlb = MultiLabelBinarizer(classes=Code_set)
    true_code_binary = code_mlb.fit_transform(true_codes)
    pred_code_binary = code_mlb.transform(pred_codes) 

    precision_code = precision_score(true_code_binary, pred_code_binary, average='micro')
    recall_code = recall_score(true_code_binary, pred_code_binary, average='micro')
    f1_code = f1_score(true_code_binary, pred_code_binary, average='micro')

    return precision_code, recall_code, f1_code

def calculate_subcode(true_sub_codes, pred_sub_codes):
    sub_code_mlb = MultiLabelBinarizer(classes=Sub_Code_set)
    true_subcode_binary = sub_code_mlb.fit_transform(true_sub_codes)
    pred_subcode_binary = sub_code_mlb.transform(pred_sub_codes)

    precision_subcode = precision_score(true_subcode_binary, pred_subcode_binary, average='micro')
    recall_subcode = recall_score(true_subcode_binary, pred_subcode_binary, average='micro')
    f1_subcode = f1_score(true_subcode_binary, pred_subcode_binary, average='micro')

    return precision_subcode, recall_subcode, f1_subcode

def calculate_jaccard_for_tokens(phrase1, phrase2):
    """
    Calculate the Jaccard coefficient of two phrases (based on tokens)
    :param phrase1: first phrase string
    :param phrase2: second phrase string
    :return: Jaccard coefficient (0~1)
    """
    set1 = set(phrase1.lower().split())
    set2 = set(phrase2.lower().split())

    # Computing intersection and union
    intersection = set1.intersection(set2)
    union = set1.union(set2)

    # Calculate the Jaccard coefficient
    if len(union) == 0:
        return 0
    return len(intersection) / len(union)


def is_full_containment_match(phrase1, phrase2):
    """
    Determine if phrase1 is completely contained in phrase2
    :param phrase1: first phrase (str) - true phrase
    :param phrase2: second phrase (str) - predicted phrase
    :return: True if phrase1 is completely contained in phrase2
    """
    set1 = set(phrase1.lower().split())
    set2 = set(phrase2.lower().split())
    
    # Determine whether set1 is a subset of set2
    return set1.issubset(set2)


def relaxed_match_evaluation_with_full_containment(true_entities_list, pred_entities_list, jaccard_threshold=0.6):
    """
    Evaluate partial matches of named entities using Relaxed Match, combining full containment logic and Jaccard coefficient
    :param true_entities_list: list of true entities, one per sentence [[str, ...], ...]
    :param pred_entities_list: list of predicted entities, one per sentence [[str, ...], ...]
    :param jaccard_threshold: Jaccard coefficient threshold for partial matches
    :return: Precision, Recall, F1 Score
    """
    true_positive = 0
    false_positive = 0
    false_negative = 0

    # Iterate over the true and predicted entities in each sentence
    for true_entities, pred_entities in zip(true_entities_list, pred_entities_list):
        matched_true = [False] * len(true_entities)
        matched_pred = [False] * len(pred_entities)

        # For each predicted entity, check whether it partially matches a real entity
        for i, pred_entity in enumerate(pred_entities):
            for j, true_entity in enumerate(true_entities):
                if not matched_true[j] and not matched_pred[i]:
                    # If the real entity is completely contained in the predicted entity, or the predicted entity is completely contained in the real entity, it is considered a match.
                    if is_full_containment_match(true_entity, pred_entity) or is_full_containment_match(pred_entity, true_entity):
                        true_positive += 1
                        matched_true[j] = True
                        matched_pred[i] = True
                    # Otherwise, use the Jaccard coefficient for partial matching evaluation.
                    elif calculate_jaccard_for_tokens(pred_entity, true_entity) >= jaccard_threshold:
                        true_positive += 1
                        matched_true[j] = True
                        matched_pred[i] = True

        # Counting False Positives and False Negatives
        false_positive += matched_pred.count(False)  # No matching predicted entities found
        false_negative += matched_true.count(False)  # No real entity matched

    # Calculate Precision, Recall, F1 Score
    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
    recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1


## loading result files

In [5]:
def parse_json(json_line):
    answer = json_line.get("target")
    response = json_line.get("filtered_resps")[0]

    return answer, response

In [6]:
def is_valid_format(input_str):
    pattern = r"""
        ^\{\s*                         # Starts with {, allows spaces before and after
        "results"\s*:\s*\[              # The "results" key, followed by [
        (\s*\{                          # Starts the JSON object
            \s*"Code"\s*:\s*'[^']+'\s*, # "Code" field, must be a non-empty string
            \s*"Sub-code"\s*:\s*'[^']+'\s*, # "Sub-code" field, must be a non-empty string
            \s*"Span"\s*:\s*'[^']+'\s*  # "Span" field, must be a non-empty string
        \}\s*,?)*                       # Multiple objects are allowed, commas are optional
        \s*\]                           # End list
        \s*\}$                          # End dict
    """

    return bool(re.match(pattern, input_str, re.VERBOSE | re.DOTALL))

def fix_structure(input_str):
    # input_str = replace_str(input_str)

    
    # Use regular expression to extract complete dictionary object
    pattern = re.compile(r'\{\s*"Code"\s*:\s*"[^"]+"\s*,\s*"Sub-code"\s*:\s*"[^"]+"\s*,\s*"Span"\s*:\s*"[^"]+"\s*\}')
    matches = pattern.findall(input_str)
    
    # Reconstruct the results list to conform to the format
    corrected_results = ', '.join(matches)
    corrected_json = '{"results": [' + corrected_results + ']}'
    
    return corrected_json

In [8]:
def evaluate_eppc_agg(items):
    true_answer = [item[0] for item in items]
    pred_answer = [item[1] for item in items]
    

    ## code
    true_codes = []
    pred_codes = []

    ## sub-code
    true_sub_codes = []
    pred_sub_codes = []

    ## span
    true_spans = []
    pred_spans = []
    for t_answer, p_answer in zip(true_answer, pred_answer):
        # t_answer = ast.literal_eval(t_answer)
        t_answer = json.loads(t_answer)
        
        if not is_valid_format(p_answer):
            p_answer = fix_structure(p_answer)
        p_answer = json.loads(p_answer)
        
        
        code = [anno.get("Code") for anno in t_answer.get("results")]
        pred_code = [pred.get("Code") for pred in p_answer.get("results")]
        true_codes.append(code)
        pred_codes.append(pred_code)

        sub_code = [anno.get("Sub-code") for anno in t_answer.get("results")]
        pred_sub_code = [pred.get("Sub-code") for pred in p_answer.get("results")]
        true_sub_codes.append(sub_code)
        pred_sub_codes.append(pred_sub_code)

        span = [anno.get("Span") for anno in t_answer.get("results")]
        extracted_span = [pred.get("Span") for pred in p_answer.get("results")]
        true_spans.append(span)
        pred_spans.append(extracted_span)

    
    precision_code, recall_code, f1_code = calculate_code(true_codes, pred_codes)
    precision_subcode, recall_subcode, f1_subcode = calculate_subcode(true_sub_codes, pred_sub_codes)
    precision_span, recall_span, f1_span = relaxed_match_evaluation_with_full_containment(true_spans, pred_spans, jaccard_threshold=0.6)
    # precision_span_gpt, recall_span_gpt, f1_span_gpt = relaxed_match_evaluation_with_semantic(true_spans, pred_spans)
    
    return {"code": {"P": round(precision_code,4), "R": round(recall_code,4), "f1": round(f1_code,4)},
            "sub-code": {"P": round(precision_subcode,4), "R": round(recall_subcode,4), "f1": round(f1_subcode,4)},
            "span": {"P": round(precision_span,4), "R": round(recall_span,4), "f1": round(f1_span,4)}
            }
def get_statistical_results(results):
    mean = np.mean(results)
    std = np.std(results, ddof=1)
    return f"{mean:.4f} ± {std:.4f}"

In [33]:
result_data_path = "/home/lm2445/AMIA2025/FinBen_Journal/results/eppc/hongzhouyu__FineMedLM-o1"
files = os.listdir(result_data_path)
files

['results_2025-08-13T23-31-00.154448.json',
 'results_2025-08-14T00-28-58.804433.json',
 'results_2025-08-14T01-26-56.734426.json',
 'results_2025-08-14T02-24-48.169420.json',
 'results_2025-08-14T03-23-10.200225.json',
 'samples_EppcExtraction_2025-08-13T23-31-00.154448.jsonl',
 'samples_EppcExtraction_2025-08-14T00-28-58.804433.jsonl',
 'samples_EppcExtraction_2025-08-14T01-26-56.734426.jsonl',
 'samples_EppcExtraction_2025-08-14T02-24-48.169420.jsonl',
 'samples_EppcExtraction_2025-08-14T03-23-10.200225.jsonl']

In [34]:
final_result = []
for file in files:
    if file.endswith("jsonl"):
        result_path = os.path.join(result_data_path,file)
        items = []
        with open(result_path, "r") as f:
            for line in tqdm(f):
                json_line = json.loads(line)
                res = parse_json(json_line)
                items.append(res)
        result = evaluate_eppc_agg(items)
        final_result.append(result)
final_result

1933it [00:00, 1967.01it/s]
/home/lm2445/.conda/envs/finben_linhai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/lm2445/.conda/envs/finben_linhai/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['AcknowledgeSituation'] will be ignored
  warnings.warn(
/home/lm2445/.conda/envs/finben_linhai/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
1933it [00:00, 1937.84it/s]
/home/lm2445/.conda/envs/finben_linhai/lib/python3.12/site-packages/

[{'code': {'P': 0.0, 'R': 0.0, 'f1': 0.0},
  'sub-code': {'P': 0.0, 'R': 0.0, 'f1': 0.0},
  'span': {'P': 0, 'R': 0.0, 'f1': 0}},
 {'code': {'P': 0.0, 'R': 0.0, 'f1': 0.0},
  'sub-code': {'P': 0.0, 'R': 0.0, 'f1': 0.0},
  'span': {'P': 0, 'R': 0.0, 'f1': 0}},
 {'code': {'P': 0.0, 'R': 0.0, 'f1': 0.0},
  'sub-code': {'P': 0.0, 'R': 0.0, 'f1': 0.0},
  'span': {'P': 0, 'R': 0.0, 'f1': 0}},
 {'code': {'P': 0.0, 'R': 0.0, 'f1': 0.0},
  'sub-code': {'P': 0.0, 'R': 0.0, 'f1': 0.0},
  'span': {'P': 0, 'R': 0.0, 'f1': 0}},
 {'code': {'P': 0.0, 'R': 0.0, 'f1': 0.0},
  'sub-code': {'P': 0.0, 'R': 0.0, 'f1': 0.0},
  'span': {'P': 0, 'R': 0.0, 'f1': 0}}]

In [35]:
statistical_result = None

codes_p = [res['code'].get('P') for res in final_result]
codes_r = [res['code'].get('R') for res in final_result]
codes_f = [res['code'].get('f1') for res in final_result]

subcodes_p = [res['sub-code'].get('P') for res in final_result]
subcodes_r = [res['sub-code'].get('R') for res in final_result]
subcodes_f = [res['sub-code'].get('f1') for res in final_result]

span_p = [res['span'].get('P') for res in final_result]
span_r = [res['span'].get('R') for res in final_result]
span_f = [res['span'].get('f1') for res in final_result]

code_p = get_statistical_results(codes_p)
code_r = get_statistical_results(codes_r)
code_f = get_statistical_results(codes_f)

subcode_p = get_statistical_results(subcodes_p)
subcode_r = get_statistical_results(subcodes_r)
subcode_f = get_statistical_results(subcodes_f)

sp_p = get_statistical_results(span_p)
sp_r = get_statistical_results(span_r)
sp_f = get_statistical_results(span_f)

statistical_result = {"code": {"P": code_p, "R": code_r, "f1": code_f},
        "sub-code": {"P": subcode_p, "R": subcode_r, "f1": subcode_f},
        "span": {"P": sp_p, "R": sp_r, "f1": sp_f}}

In [36]:
statistical_result

{'code': {'P': '0.0000 ± 0.0000',
  'R': '0.0000 ± 0.0000',
  'f1': '0.0000 ± 0.0000'},
 'sub-code': {'P': '0.0000 ± 0.0000',
  'R': '0.0000 ± 0.0000',
  'f1': '0.0000 ± 0.0000'},
 'span': {'P': '0.0000 ± 0.0000',
  'R': '0.0000 ± 0.0000',
  'f1': '0.0000 ± 0.0000'}}